In [1]:
!pip install duckdb #installing a lightweight analytical SQL database

In [2]:
import duckdb
import pandas as pd

In [3]:
data = pd.read_csv('double_pendulum_simulation_NEW.csv') #loading the CSV file

In [4]:
import os

for file in os.listdir('/content'):
  print(file)

#this was just to check if it could find the file

.config
double_pendulum_simulation_NEW.csv
sample_data


In [5]:
con = duckdb.connect('double_pendulum.duckdb') #creating a DuckDB connection

In [6]:
#turning CSV data into a SQL table
con.execute("""
    CREATE TABLE simulation AS
    SELECT * FROM data
""")

In [7]:
result = con.execute("""
    SELECT *
    FROM simulation
    LIMIT 10
""").df()

result

#this is just for testing, this will give us the first 10 rows of the table displayes as a table
#because Colab uses Python code cells, in order to use SQL, you have to write in between """

,time,theta1,theta2,omega1,omega2,x1,y1,x2,y2,kinetic_energy,potential_energy,total_energy
0,0.000000,1.570796,1.570796,0.000000,0.000000e+00,1.000000,-6.123234e-17,2.000000,-1.224647e-16,0.000000,-1.802068e-15,-1.802068e-15
1,0.004001,1.570718,1.570796,-0.039248,-2.903105e-10,1.000000,-7.851140e-05,2.000000,-7.851140e-05,0.001540,-1.540394e-03,-4.844866e-15
2,0.008002,1.570482,1.570796,-0.078496,-9.289931e-09,1.000000,-3.140456e-04,2.000000,-3.140456e-04,0.006162,-6.161575e-03,-3.160666e-15
3,0.012002,1.570090,1.570796,-0.117744,-7.054537e-08,1.000000,-7.066025e-04,2.000000,-7.066026e-04,0.013864,-1.386354e-02,-3.523223e-15
4,0.016003,1.569540,1.570796,-0.156991,-2.972771e-07,0.999999,-1.256182e-03,1.999999,-1.256182e-03,0.024646,-2.464629e-02,-5.787038e-15
5,0.020004,1.568834,1.570796,-0.196239,-9.072146e-07,0.999998,-1.962782e-03,1.999998,-1.962785e-03,0.038510,-3.850981e-02,-5.549727e-14
6,0.024005,1.567970,1.570796,-0.235486,-2.257425e-06,0.999996,-2.826401e-03,1.999996,-2.826410e-03,0.055454,-5.545408e-02,-6.407375e-14
7,0.028006,1.566949,1.570796,-0.274732,-4.879133e-06,0.999993,-3.847036e-03,1.999993,-3.847059e-03,0.075479,-7.547907e-02,-2.133987e-13
8,0.032006,1.565772,1.570796,-0.313977,-9.512510e-06,0.999987,-5.024679e-03,1.999987,-5.024730e-03,0.098585,-9.858470e-02,-3.755052e-13
9,0.036007,1.564437,1.570796,-0.353221,-1.714144e-05,0.999980,-6.359321e-03,1.999980,-6.359424e-03,0.124771,-1.247709e-01,-2.153139e-13


In [8]:
result = con.execute("""
    SELECT
    AVG(omega1) AS average_omega1,
    MAX(omega1) AS maximum_omega1,
    MIN(omega1) AS minimum_omega1
    FROM simulation
""").df()

result

#average, maximum and minimum values for the initial angle of the first pendulum

,average_omega1,maximum_omega1,minimum_omega1
0,-0.040686,6.92046,-6.431975


In [9]:
result = con.execute("""
    SELECT
    AVG(total_energy) AS average_energy,
    MAX(total_energy) AS maximum_energy,
    MIN(total_energy) AS minimum_energy
    FROM simulation
""").df()

result

#average, maximum and minimum values for the total energy

,average_energy,maximum_energy,minimum_energy
0,-3.325161e-09,1.503173e-08,-1.447068e-08


In [10]:
#at what point in the simulation does the first pendulum reach its maximum angular velocity

result = con.execute("""
    SELECT
        time,
        theta1,
        omega1
    FROM simulation
    ORDER BY ABS(omega1) DESC
    LIMIT 1
""").df()

result

,time,theta1,omega1
0,19.775955,-0.453016,6.92046


In [11]:
#divide the simulation into time periods and calculate statistics for each period

result = con.execute("""
    SELECT
        FLOOR(time / 2) AS time_period,
        AVG(ABS(omega1)) AS avg_angular_speed
    FROM simulation
    GROUP BY time_period
    ORDER BY time_period
""").df()

result

,time_period,avg_angular_speed
0,0.0,2.223554
1,1.0,2.271740
2,2.0,1.979705
3,3.0,2.268861
4,4.0,2.260998
5,5.0,2.254784
6,6.0,1.941887
7,7.0,2.394152
8,8.0,2.124037
9,9.0,3.088851


In [12]:
#checking how much data in the file

result = con.execute("""
    SELECT COUNT(*) AS row_count
    FROM simulation
""").df()

result

,row_count
0,5000


In [13]:
#checking the simulation time range

result = con.execute("""
    SELECT
        MIN(time) AS start_time,
        MAX(time) AS end_time,
        COUNT(*) AS observations
    FROM simulation
""").df()

result

,start_time,end_time,observations
0,0.0,20.0,5000


In [14]:
#How fast does each pendulum mass rotate during the simulation?

#let's find the maximum angular velocity

result = con.execute("""
    SELECT
        MAX(ABS(omega1)) AS max_angular_velocity_1,
        MAX(ABS(omega2)) AS max_angluar_velocity_2
    FROM simulation
""").df()

result

#the result gives us that the pendulum 1 has a max. angular velocity of 7.68 rad/s
#and pendulum 2 has max. angular velocity of 10.71 rad/s
#the second pendulum reaches a higher angular velocity due to the motion of the first pendulum

,max_angular_velocity_1,max_angluar_velocity_2
0,6.92046,10.687414


In [15]:
#What is the exact time of the maximum velocity of pendulum 1?

result = con.execute("""
    SELECT
        time,
        theta1,
        omega1
    FROM simulation
    ORDER BY ABS(omega1) DESC
    LIMIT 1
""").df()

result

#The negative sign on pendulum 1's velocity means it is rotating in the opposite angular direction at this instant

,time,theta1,omega1
0,19.775955,-0.453016,6.92046


In [16]:
#What is the exact time of the maximum velocity of pendulum 2?

result = con.execute("""
    SELECT
        time,
        theta2,
        omega2
    FROM simulation
    ORDER BY ABS(omega2) DESC
    LIMIT 1
""").df()

result

#Interesting - the angle of 12.805 rad is equivalent to approximately 4 full rotations
#It's still completely valid because the simulation is continuously rotating so the angles not restricted to -pi to pi

,time,theta2,omega2
0,14.466893,-6.367342,-10.687414


In [17]:
#Energy

result = con.execute("""
    SELECT
        AVG(total_energy) AS average_energy,
        MIN(total_energy) AS minimum_energy,
        MAX(total_energy) AS maximum_energy,
        MAX(total_energy) - MIN(total_energy) AS energy_range
    FROM simulation
""").df()

result

# AVG -> average total energy
# MIN -> lowest recorded energy
# MAX -> highest recorded energy
# MAX - MIN -> how much the energy varied during the simulation


#the energy range is far too large for an ideal double pendulum
#this is conservative system so total mechanical energy should remain approximately constant
#going back to python simulation notebook

#now running correctly

,average_energy,minimum_energy,maximum_energy,energy_range
0,-3.325161e-09,-1.447068e-08,1.503173e-08,2.950241e-08


In [18]:
from numpy._core.multiarray import result_type
#maximum angular displacement
#this is a measure of how far the system moves from its starting position

result = con.execute("""
    SELECT
        MAX(ABS(theta1)) AS max_angle_1,
        MAX(ABS(theta2)) AS max_angle_2
    FROM simulation
""").df()

result

,max_angle_1,max_angle_2
0,2.062508,26.121182


In [19]:
#average angular velocity
#how active each pendulum was over the simulation

result = con.execute("""
    SELECT
        AVG(ABS(omega1)) AS average_angular_velocity_1,
        AVG(ABS(omega2)) AS average_angular_velocity_2
    FROM simulation
""").df()

result

# Pendulum 2 has a higher average angular speed and reaches
#a higher peak angular speed than pendulum 1. This suggests that,
#over this simulation, pendulum 2 has been moving more rapidly.

,average_angular_velocity_1,average_angular_velocity_2
0,2.281055,3.096435


In [20]:
#investigating whether the avg. ang. speeds change over the course of the 20 sec simulation

result = con.execute("""
    SELECT
        CASE
            WHEN time < 10 THEN 'First half'
            ELSE 'Second half'
        END AS time_period,
        AVG(ABS(omega1)) AS avg_angular_velocity_1,
        AVG(ABS(omega2)) AS avg_angular_velocity_2,
        MAX(ABS(omega1)) AS max_angular_velocity_1,
        MAX(ABS(omega2)) AS max_angular_velocity_2
    FROM simulation
    GROUP BY time_period
    ORDER BY time_period DESC
""").df()

result

#CASE WHEN - puts each row into a time period

#looking at the results
#So the second half is noticeably more dynamically active, particularly for pendulum 2.
#One important clarification for your eventual README:
#this doesn't mean the pendulum is necessarily "gaining energy."
#The corrected simulation has excellent energy conservation,
#so the changing angular speeds are a consequence of the nonlinear exchange of energy between the two pendulums,
#rather than energy being created

,time_period,avg_angular_velocity_1,avg_angular_velocity_2,max_angular_velocity_1,max_angular_velocity_2
0,Second half,2.361138,3.651446,6.92046,10.687414
1,First half,2.200972,2.541425,6.05177,9.660235


In [22]:
#rechecking energy conservation

result = con.execute("""
    SELECT
        MIN(total_energy) AS minimum_energy,
        MAX(total_energy) AS maximum_energy,
        AVG(total_energy) AS average_energy,
        MAX(total_energy) - MIN(total_energy) AS energy_range
    FROM simulation
""").df()

result

,minimum_energy,maximum_energy,average_energy,energy_range
0,-1.447068e-08,1.503173e-08,-3.325161e-09,2.950241e-08
